In [ ]:
from xmlrpc.server import SimpleXMLRPCServer
from xmlrpc.server import SimpleXMLRPCRequestHandler
import threading
import datetime
import os
import pandas as pd

class RequestHandler(SimpleXMLRPCRequestHandler):
    rpc_paths = ('/RPC2',)

server = SimpleXMLRPCServer(("127.0.0.1", 8018),
                            requestHandler=RequestHandler,
                            allow_none=True)

# КОНСТАНТЫ
LOG_FILE = 'log.csv'
MAX_RECORDS = 10  # максимальное число записей в журнале


def _count_lines(path):
    #Считает количество строк в файле.
    if not os.path.isfile(path):
        return 0
    with open(path, 'r', encoding='utf8') as f:
        return sum(1 for _ in f)


def _rotate_if_needed():
    if _count_lines(LOG_FILE) >= MAX_RECORDS:
        stamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
        base, ext = os.path.splitext(LOG_FILE)      # ('log', '.csv')
        new_name = f"{base}_{stamp}{ext}"           # 'log_20260930_142233.csv'
        os.rename(LOG_FILE, new_name)


# добавление строки в лог
def add_log(sname):
    _rotate_if_needed()
    f = open(LOG_FILE, 'a', encoding='utf8')
    f.write(str(sname) + ',' + datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S") + '\n')
    f.close()
    return True

server.register_function(add_log, 'add_log')


#ПРОВЕРИТЬ ПРОВЕРИТЬ ПРОВЕРИТЬ ПРОВЕРИТЬ ПРОВЕРИТЬ ПРОВЕРИТЬ ПРОВЕРИТЬ ПРОВЕРИТЬ ПРОВЕРИТЬ ПРОВЕРИТЬ ПРОВЕРИТЬ ПРОВЕРИТЬ ПРОВЕРИТЬ ПРОВЕРИТЬ ПРОВЕРИТЬ ПРОВЕРИТЬ ПРОВЕРИТЬ ПРОВЕРИТЬ ПРОВЕРИТЬ
def get_log(event_type=None, time_from=None, time_to=None):
    files = glob("log.csv") + glob("log_*.csv")

    if not files:
        return "Log files not found"

    tables = []

    for file in files:
        table = pd.read_csv(
            file,
            header=None,
            names=["event_type", "timestamp"],
            encoding="utf8"
        )
        tables.append(table)

    # Объединяем все журналы в одну таблицу.
    df = pd.concat(tables, ignore_index=True)

    # Фильтруем уже объединённую таблицу.
    if event_type is not None:
        df = df[df["event_type"] == event_type]

    if time_from is not None:
        df = df[df["timestamp"] >= time_from]

    if time_to is not None:
        df = df[df["timestamp"] <= time_to]

    if df.empty:
        return "No records"

    # Выводим события по порядку времени.
    df = df.sort_values("timestamp")

    lines = [
        f"{row.timestamp}  {row.event_type}"
        for row in df.itertuples(index=False)
    ]
    return "\n".join(lines)


server.register_function(get_log, 'get_log')




print ("Listening on port 8018...")
server.serve_forever()


Listening on port 8018...


127.0.0.1 - - [19/Oct/2026 16:17:39] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [19/Oct/2026 16:17:39] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [19/Oct/2026 16:17:39] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [19/Oct/2026 16:17:39] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [19/Oct/2026 16:17:40] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [19/Oct/2026 16:17:44] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [19/Oct/2026 16:17:44] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [19/Oct/2026 16:17:44] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [19/Oct/2026 16:17:44] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [19/Oct/2026 16:17:45] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [19/Oct/2026 16:17:49] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [19/Oct/2026 16:20:33] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [19/Oct/2026 16:20:33] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [19/Oct/2026 16:20:33] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [19/Oct/2026 16:20:33] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [19/Oct/202